# Riconoscimento del traffico e blocco mirato dell'applicazione

## Obiettivo

Implementare una pipeline dimostrativa che, a partire da una cattura di
traffico di rete (file `.pcap`), riconosce quale applicazione ha generato tale traffico e,
se l'applicazione riconosciuta rientra nei criteri di una policy di enforcement definita
dall'amministratore, applica automaticamente un'azione di blocco a livello di rete tramite
OpenAppFilter (OAF), registrando l'azione intrapresa in un log.

La pipeline si fonda su due elementi sviluppati separatamente e qui integrati:

1. Il modello di classificazione delle applicazioni a partire da feature di traffico,
   addestrato e validato sul dataset di catture etichettate (classificazione_app.ipynb);
2. La metodologia di individuazione dei domini di rete effettivamente utilizzati da
   un'applicazione durante l'uso, impiegata per costruire regole di blocco mirate a livello
   di singola applicazione (analisi_domini.ipynb).

## Struttura

1. Configurazione dell'ambiente
2. Estrazione delle feature di traffico da una cattura
3. Addestramento del modello di classificazione finale
4. Definizione della policy di enforcement
5. Applicazione dell'azione di enforcement e registrazione del log
6. Pipeline completa
7. Esecuzione su una cattura


## 1. Configurazione dell'ambiente

In [2]:
import struct
import socket
import subprocess
import csv
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
print("Librerie caricate correttamente.")

Librerie caricate correttamente.


## 2. Estrazione delle feature di traffico da una cattura

Per classificare una cattura di traffico è necessario ridurre la sequenza di pacchetti che
la compongono a un insieme ridotto di misure numeriche (feature), calcolate secondo la
stessa definizione impiegata per costruire il dataset di addestramento del modello. Le
cinque feature utilizzate sono:

- **MB_totali**: volume complessivo di dati scambiati durante la sessione catturata.
- **rapporto_down_up**: rapporto tra byte ricevuti (download) e byte inviati (upload) dal
  dispositivo, utile a distinguere pattern di traffico dominati dal download rispetto ad
  altre tipologie di attività.
- **perc_pacchetti_pesanti**: percentuale di pacchetti che superano una soglia dimensionale
  (1200 byte), indicativa della presenza di trasferimenti di dati sostenuti rispetto a
  traffico di controllo o metadati.
- **iat_medio_ms**: tempo medio di inter-arrivo tra pacchetti consecutivi, che riflette la
  continuità e regolarità del flusso di traffico.
- **dim_std**: deviazione standard della dimensione dei pacchetti, che cattura la
  variabilità del traffico osservato.

Queste cinque feature sono il risultato di un processo di selezione condotto nella
classificazione delle applicazioni, dove sono state confrontate con configurazioni
alternative (insiemi di feature più ampi, algoritmi di classificazione diversi) tramite
validazione leave-one-out, risultando la configurazione con le migliori prestazioni di
riconoscimento.

Il calcolo di `rapporto_down_up` e `perc_pacchetti_pesanti` richiede di stabilire la
direzione di ciascun pacchetto (upload o download), determinata confrontando l'indirizzo IP
sorgente con quello del dispositivo di test sulla rete locale. Il prefisso di rete
utilizzato di default (`192.168.1.`) corrisponde alla subnet del testbed impiegato per la
raccolta dati; va modificato tramite la costante `LOCAL_IP_PREFIX` qualora la subnet della
rete di test fosse diversa.

L'estrazione avviene leggendo direttamente la struttura binaria del file `.pcap`
(un'intestazione globale di 24 byte seguita da record di lunghezza variabile, ciascuno
preceduto da un header fisso di 16 byte contenente timestamp e lunghezza del pacchetto),
senza ricorrere alla decodifica completa di ogni pacchetto: per le sole informazioni
necessarie (timestamp, dimensione, intestazione IP) è sufficiente leggere gli header e
saltare il contenuto dei pacchetti, riducendo sensibilmente i tempi di elaborazione anche su
catture di grandi dimensioni.

In [3]:
LOCAL_IP_PREFIX = "192.168.1."
SOGLIA_PACCHETTO_PESANTE = 1200  

FEATURE_COLUMNS = ["MB_totali", "rapporto_down_up", "perc_pacchetti_pesanti", "iat_medio_ms", "dim_std"]


def estrai_feature(percorso_file, local_ip_prefix=LOCAL_IP_PREFIX):
    dimensioni = []
    timestamps = []
    upload_bytes = 0
    download_bytes = 0
    n_pesanti = 0
    n_leggeri = 0

    with open(percorso_file, "rb") as f:
        header_globale = f.read(24)
        if len(header_globale) < 24:
            return None

        magic = struct.unpack("I", header_globale[:4])[0]
        if magic == 0xa1b2c3d4:
            endian = "<"
        elif magic == 0xd4c3b2a1:
            endian = ">"
        else:
            return None

        while True:
            header_pacchetto = f.read(16)
            if len(header_pacchetto) < 16:
                break
            ts_sec, ts_usec, len_catturata, len_originale = struct.unpack(
                endian + "IIII", header_pacchetto
            )
            dati_pacchetto = f.read(len_catturata)

            timestamps.append(ts_sec + ts_usec / 1_000_000)
            dimensioni.append(len_originale)

            if len(dati_pacchetto) >= 34:
                ip_header = dati_pacchetto[14:34]
                if ip_header[0] >> 4 == 4:  # IPv4
                    ip_src = socket.inet_ntoa(ip_header[12:16])
                    if ip_src.startswith(local_ip_prefix):
                        upload_bytes += len_originale
                    else:
                        download_bytes += len_originale

                    if len_originale > SOGLIA_PACCHETTO_PESANTE:
                        n_pesanti += 1
                    else:
                        n_leggeri += 1

    if not dimensioni:
        return None

    n_pacchetti = len(dimensioni)
    byte_totali = sum(dimensioni)
    durata_sec = timestamps[-1] - timestamps[0]
    arr = np.array(dimensioni)
    iat = np.diff(timestamps)
    totale_direzionale = n_pesanti + n_leggeri

    return {
        "n_pacchetti": n_pacchetti,
        "MB_totali": round(byte_totali / (1024 * 1024), 2),
        "durata_sec": round(durata_sec, 2),
        "dim_pacchetto_media": round(byte_totali / n_pacchetti, 1),
        "dim_std": round(float(arr.std()), 1),
        "upload_MB": round(upload_bytes / (1024 * 1024), 2),
        "download_MB": round(download_bytes / (1024 * 1024), 2),
        "rapporto_down_up": round(download_bytes / upload_bytes, 2) if upload_bytes > 0 else None,
        "perc_pacchetti_pesanti": round(100 * n_pesanti / totale_direzionale, 1) if totale_direzionale > 0 else None,
        "iat_medio_ms": round(float(np.mean(iat)) * 1000, 3) if len(iat) > 0 else None,
    }


def estrai_vettore_feature(percorso_file, local_ip_prefix=LOCAL_IP_PREFIX):
    feat = estrai_feature(percorso_file, local_ip_prefix=local_ip_prefix)
    if feat is None:
        return None
    valori = [feat[c] for c in FEATURE_COLUMNS]
    if any(v is None for v in valori):
        return None
    return valori


print("Funzioni di estrazione feature pronte.")

Funzioni di estrazione feature pronte.


## 3. Addestramento del modello di classificazione finale

Il modello impiegato è un Random Forest (`n_estimators=200`), addestrato sulle cinque
feature descritte nella sezione precedente. Questa configurazione corrisponde a quella
individuata come migliore nella classificazione delle applicazioni citata in
apertura, dove è stata confrontata con configurazioni alternative tramite validazione
leave-one-out, ottenendo un'accuratezza dell'83.3% nella distinzione tra nove applicazioni.

In questa pipeline il modello viene addestrato sull'intero insieme di campioni disponibili,
anziché essere ri-addestrato a ogni iterazione della validazione incrociata: l'obiettivo qui
non è stimare un'accuratezza, ma ottenere un classificatore pronto per essere applicato a
catture nuove, non presenti nel dataset di addestramento. Prima di procedere, viene comunque
ripetuta la validazione leave-one-out come controllo di coerenza tra il dataset caricato e
quello impiegato nell'analisi originale: un'accuratezza prossima all'83.3% conferma che il
file caricato corrisponde al dataset atteso.

In [4]:
CSV_TRAINING = "dati/statistiche_complete_app.csv" 

df = pd.read_csv(CSV_TRAINING)

conteggi = df["app"].value_counts()
app_valide = conteggi[conteggi >= 2].index.tolist()
df_filtrato = df[df["app"].isin(app_valide)].copy()

X = df_filtrato[FEATURE_COLUMNS].fillna(0).values
y = df_filtrato["app"].values

loo = LeaveOneOut()
predette, reali = [], []
for train_idx, test_idx in loo.split(X):
    clf = RandomForestClassifier(n_estimators=200, random_state=42)
    clf.fit(X[train_idx], y[train_idx])
    predette.append(clf.predict(X[test_idx])[0])
    reali.append(y[test_idx][0])

acc = accuracy_score(reali, predette)
print(f"Verifica leave-one-out: {acc*100:.1f}%")

modello = RandomForestClassifier(n_estimators=200, random_state=42)
modello.fit(X, y)

classi = sorted(app_valide)
print(f"Modello allenato. Classi riconosciute ({len(classi)}): {classi}")

joblib.dump({"modello": modello, "classi": classi, "feature_columns": FEATURE_COLUMNS}, "dati/modello_finale.joblib")
print("Modello salvato in modello_finale.joblib")

Verifica leave-one-out: 83.3%
Modello allenato. Classi riconosciute (9): ['Calcolatrice', 'Canva', 'Corriere della Sera', 'Disney+', 'IKEA', 'Shazam', 'Trivago', 'UPS', 'Unieuro']
Modello salvato in modello_finale.joblib


## 4. Definizione della policy di enforcement

Le azioni di enforcement disponibili hanno due livelli di granularità diversi:

- **Blocco mirato per applicazione (per-app)**: basato su firme di dominio specifiche del
  servizio associato all'applicazione, individuate tramite l'analisi dei domini contattati
  durante l'uso reale dell'applicazione (analisi_domini.ipynb). Questo
  tipo di blocco è stato verificato empiricamente solo per le applicazioni elencate in
  `POLICY_PER_APP`, per le quali il dominio proprietario è risultato sufficientemente
  consolidato, cioè non disperso tra servizi di terze parti (SDK di analytics, A/B
  testing, pagamento), da consentire un blocco efficace del funzionamento
  dell'applicazione.
- **Blocco a livello di store applicativo**: basato sulle firme di dominio degli store
  (AppStore, GooglePlay) già presenti nel set di regole di OpenAppFilter. Blocca l'accesso
  allo store, non il funzionamento dell'applicazione già installata; è impiegato come
  azione di ripiego per le applicazioni che rientrano nella policy di interesse ma per cui
  non è stato ancora verificato un blocco per app affidabile.

La policy è rappresentata da due strutture: un dizionario che associa a ciascuna
applicazione verificata l'elenco degli identificativi di regola OAF corrispondenti al
proprio dominio, e una lista di applicazioni di interesse per le quali si ricade sul blocco
di store in assenza di una regola per app verificata.

Le variabili che seguono definiscono concretamente la policy:

- `POLICY_PER_APP`: dizionario che associa a ciascuna applicazione con blocco per-app verificato l'elenco degli identificativi di regola OAF corrispondenti ai domini che le sono stati associati.
- `POLICY_STORE_FALLBACK`: elenco delle applicazioni di interesse per cui si applica il blocco di store in assenza di una regola per-app verificata.
- `STORE_IDS`: identificativi delle regole OAF corrispondenti agli store applicativi (AppStore, GooglePlay), utilizzati per il blocco di ripiego.
- `ROUTER_HOST`: indirizzo del router OpenWrt su cui viene eseguita l'azione di blocco, nella forma utente@indirizzo IP.
- `OAF_BLOCK_IDS_SCRIPT_REMOTO`: percorso, sul router, dello script che riceve in ingresso gli identificativi da bloccare e applica la configurazione a OpenAppFilter.
- `DRY_RUN`: se `True`, la pipeline determina e mostra l'azione che verrebbe eseguita senza inviarla al router; va impostato a `False` per applicarla realmente.

Per l'applicazione con blocco per app verificato, sono stati registrati sei identificativi di regola, uno per ciascun dominio individuato nell'analisi dei domini; nel test empirico non tutti sono risultati necessari a produrre il blocco osservato, a indicazione di una parziale ridondanza tra le firme registrate, mantenute comunque nella policy per coprire funzionalità dell'applicazione diverse da quella osservata durante il test.

In [6]:
POLICY_PER_APP = {
    "IKEA": ["9010", "9011", "9012", "9013", "9014", "9015"],
}

POLICY_STORE_FALLBACK = ["Canva", "Disney+"]

STORE_IDS = ["7002", "9001"]  

ROUTER_HOST = "root@192.168.1.1"                    
OAF_BLOCK_IDS_SCRIPT_REMOTO = "/root/oaf_block_ids.sh" 

DRY_RUN = True   

print(f"Blocco per-app verificato per: {list(POLICY_PER_APP.keys())}")
print(f"Blocco store-level (ripiego) per: {POLICY_STORE_FALLBACK}")
print(f"DRY_RUN = {DRY_RUN}")

Blocco per-app verificato per: ['IKEA']
Blocco store-level (ripiego) per: ['Canva', 'Disney+']
DRY_RUN = True


## 5. Applicare l'enforcement via SSH e con log di riferimento

L'attuazione delle contromisure di rete si articola attraverso la mappatura delle policy, l'esecuzione remota e la persistenza dei dati:
* `decidi_azione`: Mappa l'applicazione riconosciuta alla strategia di blocco appropriata (per app o store-level) in base alla policy definita.
* `applica_enforcement`: Coordina l'esecuzione remota tramite il modulo `subprocess`. L'uso di `subprocess.run` consente di invocare comandi SSH in modo sincrono, catturando i flussi `stdout` e `stderr` per la diagnostica immediata. La gestione del `returncode` e l'impostazione di un `timeout` (15s)  evitano il blocco della pipeline in caso di latenze o errori di connessione sul router.
* `registra_log`: Assicura l'archiviazione persistente di ogni operazione in un file CSV, garantendo l'auditability del sistema e permettendo analisi retrospettive sugli esiti dei comandi inviati.

In [10]:
LOG_PATH = "dati/log_enforcement.csv"

def decidi_azione(app_riconosciuta):
    if app_riconosciuta in POLICY_PER_APP:
        return POLICY_PER_APP[app_riconosciuta], "blocco per-app (dominio specifico, verificato)"
    elif app_riconosciuta in POLICY_STORE_FALLBACK:
        return STORE_IDS, "blocco store-level (nessun blocco per-app verificato per questa app)"
    else:
        return None, "nessuna"


def applica_enforcement(ids, dry_run=DRY_RUN):
    comando = ["ssh", ROUTER_HOST, OAF_BLOCK_IDS_SCRIPT_REMOTO] + ids

    if dry_run:
        print(f"[DRY RUN] Comando che verrebbe eseguito: {' '.join(comando)}")
        return "dry_run (nessun comando eseguito realmente)"

    try:
        risultato = subprocess.run(comando, capture_output=True, text=True, timeout=15)
        if risultato.returncode == 0:
            return f"eseguito con successo: {risultato.stdout.strip()}"
        else:
            return f"errore (exit {risultato.returncode}): {risultato.stderr.strip()}"
    except Exception as e:
        return f"errore di connessione SSH: {e}"


def registra_log(percorso_pcap, app_riconosciuta, confidenza, azione, ids, esito):
    file_esiste = Path(LOG_PATH).exists()
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_esiste:
            writer.writerow(["timestamp", "cattura", "app_riconosciuta", "confidenza",
                              "azione", "id_bloccati", "esito"])
        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            percorso_pcap, app_riconosciuta, f"{confidenza:.2f}" if confidenza else "",
            azione, " ".join(ids) if ids else "", esito,
        ])

print("Funzioni di decisione, enforcement e logging pronte.")

Funzioni di decisione, enforcement e logging pronte.


## 6. Pipeline completa

In [7]:
def classifica_cattura(percorso_pcap):
    vettore = estrai_vettore_feature(percorso_pcap)
    if vettore is None:
        return None, None
    predizione = modello.predict([vettore])[0]
    probabilita = modello.predict_proba([vettore])[0]
    confidenza = max(probabilita)
    return predizione, confidenza


def esegui_pipeline(percorso_pcap):
    print(f"--- Analisi cattura: {percorso_pcap} ---")

    app_riconosciuta, confidenza = classifica_cattura(percorso_pcap)
    if app_riconosciuta is None:
        print("Cattura vuota o illeggibile, nessuna azione.")
        registra_log(percorso_pcap, "N/D", None, "nessuna", None, "cattura illeggibile")
        return

    print(f"App riconosciuta: {app_riconosciuta}  (confidenza: {confidenza:.2f})")

    ids, azione = decidi_azione(app_riconosciuta)

    if ids is None:
        print(f"'{app_riconosciuta}' non e' nella lista di interesse: nessuna azione.")
        esito = "nessuna azione necessaria"
    else:
        print(f"Azione: {azione}")
        esito = applica_enforcement(ids)

    registra_log(percorso_pcap, app_riconosciuta, confidenza, azione, ids, esito)
    print(f"Esito: {esito}")
    print(f"Log aggiornato: {LOG_PATH}")

print("Pipeline pronta.")

Pipeline pronta.


## 7. Esecuzione su una cattura

Percorso di una cattura di traffico (`.pcap`) da classificare. La pipeline
restituisce l'applicazione riconosciuta, la confidenza della predizione, l'azione di
enforcement decisa in base alla policy definita, e l'esito
dell'applicazione dell'azione.

In [ ]:
esegui_pipeline("catture/cattura_ikea_1.pcap") 